In [22]:
# imports
import pandas as pd
import numpy as np
import os
import json

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report
)

In [4]:
# Load data
df = pd.read_csv("../data/processed/wesad_hrv_features.csv")
df.shape

(865, 94)

In [5]:
# Fixed feature columns
FEATURE_COLS = [
    "HRV_MeanNN",
    "HRV_SDNN",
    "HRV_RMSSD",
    "HRV_pNN50",
    "HRV_LF",
    "HRV_HF",
    "HRV_LFHF"
]

In [7]:
# Labels and groups
X = df[FEATURE_COLS]
y = (df["label"] == 2).astype(int)  # 1 = stress
groups = df["subject"]

In [9]:
# Cross-validation setup
gkf = GroupKFold(n_splits=5)

# Logistic Regression Baseline Model

In [10]:
logreg_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000))
])

In [11]:
# Cross validation evaluation
logreg_results = []

for train_idx, test_idx in gkf.split(X, y, groups):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    logreg_pipeline.fit(X_train, y_train)
    y_pred = logreg_pipeline.predict(X_test)
    y_prob = logreg_pipeline.predict_proba(X_test)[:, 1]

    logreg_results.append({
        "accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob)
    })

pd.DataFrame(logreg_results).mean()


accuracy    0.804624
f1          0.709402
roc_auc     0.870252
dtype: float64

In [14]:
# Feature importance
coef_df = pd.DataFrame({
    "feature": FEATURE_COLS,
    "coefficient": logreg_pipeline.named_steps["clf"].coef_[0]
}).sort_values("coefficient", key=abs, ascending=False)

coef_df

,feature,coefficient
0,HRV_MeanNN,-3.693706
1,HRV_SDNN,1.051831
4,HRV_LF,0.673512
3,HRV_pNN50,0.641131
6,HRV_LFHF,-0.520056
5,HRV_HF,-0.200439
2,HRV_RMSSD,0.180196


# Non-Linear Baseline Model - Random Forest

In [12]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    random_state=4
)

In [13]:
# Cross validation evaluation
rf_results = []

for train_idx, test_idx in gkf.split(X, y, groups):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    y_prob = rf.predict_proba(X_test)[:, 1]

    rf_results.append({
        "accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob)
    })

pd.DataFrame(rf_results).mean()


accuracy    0.796532
f1          0.730003
roc_auc     0.820764
dtype: float64

In [15]:
# Feature importance
rf_importance = pd.DataFrame({
    "feature": FEATURE_COLS,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

rf_importance

,feature,importance
0,HRV_MeanNN,0.461988
3,HRV_pNN50,0.169417
2,HRV_RMSSD,0.138945
1,HRV_SDNN,0.087803
5,HRV_HF,0.054923
6,HRV_LFHF,0.052757
4,HRV_LF,0.034168


In [ ]:
# Save results
os.makedirs("../runs", exist_ok=True)
coef_df.to_csv("../runs/logreg_coefficients.csv", index=False)
rf_importance.to_csv("../runs/rf_importance.csv", index=False)

In [19]:
from joblib import dump 

os.makedirs("../runs/models", exist_ok=True) 
dump(logreg_pipeline, "../runs/models/logreg_baseline.joblib")

['../runs/models/logreg_baseline.joblib']

In [21]:
dump(rf, "../runs/models/rf_baseline.joblib")

['../runs/models/rf_baseline.joblib']

In [23]:
# Save feature list
with open("../runs/models/hrv_feature_list.json", "w") as f:
    json.dump(FEATURE_COLS, f)

In [25]:
metrics = {
    "logreg": logreg_results,
    "random_forest": rf_results
}

dump(metrics, "../runs/models/baseline_metrics.joblib")

['../runs/models/baseline_metrics.joblib']